# Fine-Tuning SPLADE on Amazon ESCI: Self-Study Notebook

This notebook is the companion to `notebooks/lab.ipynb`. The lab shows how the finished model behaves in Qdrant; this notebook shows the training recipe behind that kind of model.

It follows the core path from Qdrant's Part 2 article: ESCI query-product pairs -> product text formatting -> `SparseEncoder` -> `SpladeLoss` -> checkpoint. The article runs the job on Modal; this notebook keeps the code local so it is easier to read and adapt. Use a GPU for real training.

## Run Modes

- `RUN_TRAINING = False` by default: the notebook prepares the data and skips the expensive model fit safely.
- Set `RUN_TRAINING = True` after installing the training extras and selecting a GPU runtime.
- `LOAD_EXISTING_MODEL` reloads `out/splade-ecommerce-esci/final` automatically if you already trained once.
- `TRAIN_ROW_LIMIT`, `TEST_ROW_LIMIT`, `MAX_TRAIN_PAIRS`, and `MAX_EVAL_QUERIES` default to bounded home-run values. Set the row limits to `None` when you move to an article-scale training job.

## What You'll See

1. Why SPLADE fine-tuning helps product search
2. Setup and run modes
3. ESCI loading and product text formatting
4. Article-style anchor/positive training data
5. A runnable `SparseEncoderTrainer` setup with `SpladeLoss`
6. A small held-out eval loop using the workshop metrics
7. Active-term inspection after training
8. Optional hard-negative mining as the next improvement
9. How to adapt the recipe to your own catalog
10. Pitfalls and evaluation discipline
11. Where to go next

Source article: https://qdrant.tech/articles/sparse-embeddings-ecommerce-part-2/

## 1. Why Fine-Tune SPLADE on ESCI

A SPLADE encoder trained on general web text is competent at broad passage retrieval, but product search has a different shape:

- **Short, attribute-heavy queries:** `red dress size medium`
- **Brand and model sensitivity:** `iPhone 15 Pro Max 256GB`
- **Exact vs Substitute:** a 128GB phone is related to a 256GB phone, but it is not the product the shopper asked for

Fine-tuning teaches the model which lexical expansions help in e-commerce and which expansions create subtle product mistakes. The main lab measures this against BM25 and generic dense retrieval on held-out ESCI queries.

## 2. Setup

The main lab dependencies are enough to read this notebook. To run the training cells, install the training extras from the README:

```bash
python -m pip install "sentence-transformers>=5,<6" "accelerate>=1,<2" "huggingface_hub>=0.36,<1"
```

The code below deliberately keeps the expensive work behind flags. This makes the notebook safe to run top-to-bottom while still giving you the exact cells to turn on for a real run.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch
from datasets import Dataset, load_dataset
from IPython.display import display, Markdown
from tqdm import tqdm

# Make local helpers importable whether the notebook starts in repo root or notebooks/.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from eval.metrics import ndcg_at_k, recall_at_k

# Keep defaults safe for a laptop. Flip RUN_TRAINING only when you have the
# training extras installed and are ready to download/train the model.
RUN_TRAINING = False
RUN_OPTIONAL_HARD_NEGATIVES = False

DATASET_ID = "tasksource/esci"
LOCALE = "us"
BASE_MODEL = "distilbert/distilbert-base-uncased"
OUTPUT_DIR = REPO_ROOT / "out" / "splade-ecommerce-esci"
TRAINED_MODEL_PATH = OUTPUT_DIR / "final"
LOAD_EXISTING_MODEL = TRAINED_MODEL_PATH.exists()

# Defaults are bounded for home runs. Set row limits to None and increase
# MAX_TRAIN_PAIRS when you move to an article-scale GPU job.
TRAIN_ROW_LIMIT = 50_000
TEST_ROW_LIMIT = 10_000
MAX_TRAIN_PAIRS = 100_000
MAX_EVAL_QUERIES = 200
TRAIN_EPOCHS = 1  # Use 3 for a real GPU run once the mechanics are validated.
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Training enabled: {RUN_TRAINING}")
print(f"Existing checkpoint loading enabled: {LOAD_EXISTING_MODEL}")

## 3. Load ESCI and Format Product Text

The article calls out product text formatting because SPLADE is lexically grounded: the tokens you give the model become the vocabulary dimensions it can activate. Dense models can blur fields together; sparse models benefit from keeping brand, title, description, and bullets distinct.

This helper mirrors the article's pattern, but stays defensive because dataset mirrors do not always expose the same optional columns.

In [ ]:
def normalize_grade(label):
    """Return the ESCI grade letter: E, S, C, or I."""
    return str(label)[0].upper() if label else "I"


def clean_text(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return " | ".join(str(v).strip() for v in value if str(v).strip())
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def build_product_text(row, max_chars=512):
    """Build the product text that SPLADE will learn to match against."""
    brand = clean_text(row.get("product_brand", ""))
    title = clean_text(row.get("product_title", ""))
    color = clean_text(row.get("product_color", ""))
    description = clean_text(row.get("product_description", ""))
    bullets = clean_text(row.get("product_bullet_point", ""))

    parts = []
    if brand:
        parts.append(f"[{brand}]")
    if title:
        parts.append(title)
    if color:
        parts.append(f"| color: {color}")
    if description:
        parts.append(f"| {description[:200]}")
    if bullets:
        parts.append(f"| {bullets[:200]}")

    return " ".join(parts)[:max_chars]


print("Loading ESCI from Hugging Face...")
raw_train = load_dataset(DATASET_ID, split="train")
raw_test = load_dataset(DATASET_ID, split="test")

train_df = pd.DataFrame(raw_train)
test_df = pd.DataFrame(raw_test)

train_df = train_df[train_df["product_locale"].str.lower() == LOCALE].copy()
test_df = test_df[test_df["product_locale"].str.lower() == LOCALE].copy()

if TRAIN_ROW_LIMIT is not None:
    train_df = train_df.head(TRAIN_ROW_LIMIT).copy()
if TEST_ROW_LIMIT is not None:
    test_df = test_df.head(TEST_ROW_LIMIT).copy()

for df in (train_df, test_df):
    df["grade"] = df["esci_label"].map(normalize_grade)
    df["product_text"] = df.apply(build_product_text, axis=1)

print(f"train rows: {len(train_df):,}")
print(f"test rows:  {len(test_df):,}")
display(train_df[["query", "product_title", "grade", "product_text"]].head(3))

## 4. Build Article-Style Training Pairs

Part 2 trains from `(anchor, positive)` pairs: the anchor is the query, and the positive is an `E` or `S` product. `SparseMultipleNegativesRankingLoss` treats the other positives in the batch as in-batch negatives, while `SpladeLoss` adds the sparsity regularization that keeps vectors usable for retrieval.

We split by `query_id`, not by row, so the same query does not appear in both train and validation.

In [ ]:
POSITIVE_GRADES = {"E", "S"}

positive_df = train_df[train_df["grade"].isin(POSITIVE_GRADES)].copy()
positive_df = positive_df.drop_duplicates(subset=["query_id", "product_id"])

val_qids = positive_df["query_id"].drop_duplicates().sample(frac=0.05, random_state=42)
val_pairs_df = positive_df[positive_df["query_id"].isin(val_qids)].copy()
train_pairs_df = positive_df[~positive_df["query_id"].isin(val_qids)].copy()

if MAX_TRAIN_PAIRS and len(train_pairs_df) > MAX_TRAIN_PAIRS:
    train_pairs_df = train_pairs_df.sample(n=MAX_TRAIN_PAIRS, random_state=42).copy()

train_pairs_df = train_pairs_df.rename(columns={"query": "anchor", "product_text": "positive"})
val_pairs_df = val_pairs_df.rename(columns={"query": "anchor", "product_text": "positive"})

train_dataset = Dataset.from_pandas(train_pairs_df[["anchor", "positive"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_pairs_df[["anchor", "positive"]], preserve_index=False)

print(f"train pairs: {len(train_dataset):,}")
print(f"val pairs:   {len(val_dataset):,}")
display(train_pairs_df[["anchor", "positive", "grade"]].head(3))

## 5. Train SPLADE

This is the local version of the article's training function. The important pieces are:

- `SparseEncoder(BASE_MODEL)`: a masked-language-model backbone plus SPLADE pooling
- `SparseMultipleNegativesRankingLoss`: pulls matched query/product pairs together and uses the rest of the batch as negatives
- `SpladeLoss`: wraps the ranking loss with sparsity regularization
- `SparseEncoderTrainer`: the standard trainer for sparse encoders

The article uses Modal for GPU execution and persistent checkpoints. This notebook writes checkpoints to `out/splade-ecommerce-esci/` instead.

In [ ]:
trained_model = None

try:
    from sentence_transformers import (
        SparseEncoder,
        SparseEncoderTrainer,
        SparseEncoderTrainingArguments,
    )
except ModuleNotFoundError as exc:
    if RUN_TRAINING or (LOAD_EXISTING_MODEL and TRAINED_MODEL_PATH.exists()):
        raise ModuleNotFoundError(
            "Install training extras first: python -m pip install "
            "\"sentence-transformers>=5,<6\" \"accelerate>=1,<2\" \"huggingface_hub>=0.36,<1\""
        ) from exc
    SparseEncoder = SparseEncoderTrainer = SparseEncoderTrainingArguments = None

if not RUN_TRAINING:
    if LOAD_EXISTING_MODEL and TRAINED_MODEL_PATH.exists():
        trained_model = SparseEncoder(str(TRAINED_MODEL_PATH), device=DEVICE)
        print(f"Loaded existing checkpoint from {TRAINED_MODEL_PATH}")
    else:
        print("Skipping training. Set RUN_TRAINING = True in the setup cell to run this section.")
else:
    from sentence_transformers.sparse_encoder.losses import (
        SparseMultipleNegativesRankingLoss,
        SpladeLoss,
    )
    from sentence_transformers.training_args import BatchSamplers

    trained_model = SparseEncoder(
        BASE_MODEL,
        model_kwargs={"dtype": torch.float32},
        device=DEVICE,
    )

    loss = SpladeLoss(
        model=trained_model,
        loss=SparseMultipleNegativesRankingLoss(model=trained_model),
        query_regularizer_weight=5e-5,
        document_regularizer_weight=3e-5,
    )

    args = SparseEncoderTrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=TRAIN_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        fp16=(DEVICE == "cuda"),
        bf16=False,
        dataloader_pin_memory=(DEVICE == "cuda"),
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        eval_strategy="steps",
        eval_steps=1_000,
        save_strategy="steps",
        save_steps=1_000,
        save_total_limit=2,
        logging_steps=100,
        run_name="splade-ecommerce-esci-local",
    )

    trainer = SparseEncoderTrainer(
        model=trained_model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        loss=loss,
    )
    trainer.train()

    trained_model.save_pretrained(str(TRAINED_MODEL_PATH))
    print(f"Saved model to {TRAINED_MODEL_PATH}")

## 6. Evaluate on Held-Out ESCI Queries

This is a small in-memory evaluation, not the full Qdrant indexing path from the main lab. It answers a narrower question: after training, does this model retrieve graded products for held-out ESCI queries better than chance?

Treat the absolute numbers as a mechanics check. To prove quality, compare the trained model against BM25 and a generic dense baseline on the same held-out query set, as `notebooks/lab.ipynb` does.

In [ ]:
def build_eval_subset(test_frame, max_queries=200, seed=13):
    query_ids = test_frame["query_id"].drop_duplicates().astype(int)
    if len(query_ids) > max_queries:
        eval_qids = query_ids.sample(n=max_queries, random_state=seed).tolist()
    else:
        eval_qids = query_ids.tolist()
    rows = test_frame[test_frame["query_id"].astype(int).isin(eval_qids)].copy()
    products = rows.drop_duplicates("product_id")[["product_id", "product_text"]].reset_index(drop=True)

    queries = []
    for qid, group in rows.groupby("query_id"):
        queries.append({
            "query_id": int(qid),
            "query": group.iloc[0]["query"],
            "qrels": dict(zip(group["product_id"], group["grade"])),
        })
    return products, queries


def evaluate_sparse_encoder(model, product_frame, queries, k=10, batch_size=32):
    product_ids = product_frame["product_id"].tolist()
    product_texts = product_frame["product_text"].tolist()

    doc_embeddings = model.encode_document(
        product_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=True,
    )
    query_texts = [item["query"] for item in queries]
    query_embeddings = model.encode_query(
        query_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=True,
    )
    score_matrix = model.similarity(query_embeddings, doc_embeddings)

    rows = []
    for row_idx, item in enumerate(tqdm(queries, desc="eval", unit="q")):
        scores = score_matrix[row_idx]
        top_idx = scores.argsort(descending=True)[:k].detach().cpu().tolist()
        retrieved = [product_ids[i] for i in top_idx]
        rows.append({
            "query_id": item["query_id"],
            "query": item["query"],
            "nDCG@10": ndcg_at_k(retrieved, item["qrels"], k=10),
            "Recall@10": recall_at_k(retrieved, item["qrels"], k=10),
        })
    return pd.DataFrame(rows)


eval_products, eval_queries = build_eval_subset(test_df, max_queries=MAX_EVAL_QUERIES)
print(f"eval queries: {len(eval_queries):,}")
print(f"eval products: {len(eval_products):,}")

if trained_model is None:
    print("Skipping model eval because training did not run.")
else:
    eval_df = evaluate_sparse_encoder(
        trained_model,
        eval_products,
        eval_queries,
        batch_size=EVAL_BATCH_SIZE,
    )
    display(eval_df[["nDCG@10", "Recall@10"]].mean().to_frame("mean").T)
    display(eval_df.sort_values("nDCG@10", ascending=False).head(5))

## 7. Inspect Active Terms

Sparse vectors are inspectable. After training, look at the query terms the model activates and make sure product-critical terms survive: model numbers, quantities, sizes, colors, and brands.

In [ ]:
def sparse_tensor_to_tuple(vec):
    """Convert a SparseEncoder tensor for one text into (indices, values)."""
    if isinstance(vec, list):
        vec = vec[0]
    if hasattr(vec, "dim") and vec.dim() == 2:
        vec = vec[0]
    if getattr(vec, "is_sparse", False):
        coo = vec.coalesce()
        return coo.indices()[-1].cpu().tolist(), coo.values().detach().cpu().tolist()

    nz = torch.nonzero(vec, as_tuple=False).squeeze(-1)
    return nz.cpu().tolist(), vec[nz].detach().cpu().tolist()


if trained_model is None:
    print("Skipping active-term inspection because training did not run.")
else:
    from eval.viewer import inspect_sparse_vector

    query_text = "iphone 15 pro max 256gb"
    sparse_vec = trained_model.encode_query(query_text, convert_to_tensor=True)

    tokenizer = getattr(trained_model, "tokenizer", None)
    if tokenizer is None and hasattr(trained_model, "get_tokenizer"):
        tokenizer = trained_model.get_tokenizer()
    vocab = {int(tid): token for token, tid in tokenizer.vocab.items()}

    display(Markdown(f"Sparse terms for `{query_text}`"))
    _ = inspect_sparse_vector(sparse_tensor_to_tuple(sparse_vec), vocab=vocab, top_k=15)

## 8. Optional: Mine Hard Negatives

Part 2 of the article keeps training simple with anchor/positive pairs and in-batch negatives. The natural next improvement is hard-negative mining: find products that look close to the query but are not judged relevant, then train the model to separate them.

Keep this optional. It adds compute and pipeline complexity, but it is often the biggest quality lever once the basic training loop works. The cell below only demonstrates mining candidates; to train with them, build a new dataset with `anchor`, `positive`, and `negative` columns and pass that dataset to the same trainer.

In [ ]:
if not RUN_OPTIONAL_HARD_NEGATIVES:
    print("Skipping hard-negative mining. Set RUN_OPTIONAL_HARD_NEGATIVES = True to run this section.")
else:
    from sentence_transformers import SentenceTransformer
    import numpy as np

    product_catalog = train_df.drop_duplicates("product_id")[["product_id", "product_text"]].reset_index(drop=True)
    dense_miner = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    product_embeddings = dense_miner.encode(
        product_catalog["product_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    relevant_by_qid = (
        train_df[train_df["grade"].isin({"E", "S", "C"})]
        .groupby("query_id")["product_id"]
        .apply(set)
        .to_dict()
    )

    def mine_hard_negatives(query, relevant_product_ids, k=8, top_n=100):
        query_embedding = dense_miner.encode(query, normalize_embeddings=True)
        scores = product_embeddings @ query_embedding
        candidate_count = min(max(int(top_n), 0), len(product_catalog))
        if candidate_count == 0:
            return []
        candidate_idx = np.argpartition(-scores, candidate_count - 1)[:candidate_count]
        candidates = [
            i for i in candidate_idx
            if product_catalog.iloc[i]["product_id"] not in relevant_product_ids
        ]
        return candidates[:k]

    sample = train_df[train_df["grade"].isin({"E", "S"})].iloc[0]
    negatives = mine_hard_negatives(
        sample["query"],
        relevant_by_qid.get(sample["query_id"], set()),
    )
    display(Markdown(f"Hard negatives for `{sample['query']}`"))
    display(product_catalog.iloc[negatives][["product_id", "product_text"]])

## 9. Applying This to Your Own Catalog

The loop is the same outside ESCI:

1. Build an eval set before training. Split by query, not by row.
2. Format product text consistently. Keep brand, title, attributes, and description distinct.
3. Start with anchor/positive pairs from clicks, conversions, or human labels.
4. Train with `SpladeLoss` and inspect active terms.
5. Compare against BM25 and a generic dense baseline on held-out queries.
6. Add hard negatives only after the basic loop is working.
7. Serve the trained model through Qdrant, then decide whether hybrid retrieval or reranking adds more lift.

Useful label sources:

- Click logs for broad weak positives
- Add-to-cart or purchase logs for cleaner positives
- Manual annotation for a trusted held-out eval set
- Search-team review for the highest-value query segments

For a real training run, use the bounded defaults first to validate mechanics, then remove row caps, increase the number of pairs, and use about three epochs before comparing against the main lab baselines.

## 10. Pitfalls and Evaluation Discipline

Before trusting a fine-tune, check the failure modes that most often make retrieval experiments look better than they are:

- **Leakage:** split by `query_id`, not by row, so the same query cannot appear in train and validation.
- **Weak baseline frame:** report lift against BM25 and a generic dense model, not just absolute nDCG/recall for the trained model.
- **Catastrophic forgetting:** domain fine-tuning can make the model less useful for general-domain or mixed-intent traffic. Route non-product queries elsewhere, train per-domain models, or use a multi-domain/adapted setup if your search box serves many intents.
- **Thin labels:** a few examples can prove the code path, but reliable quality work needs enough query-product judgments to cover brands, attributes, head queries, and long-tail phrasing.
- **Catalog drift:** recheck terms and metrics after catalog, taxonomy, or merchandising changes.
- **Hard negatives too early:** mine hard negatives after the simple anchor/positive path works and after you have a baseline to beat.

## 11. Where To Go Next

- **Article Part 2:** full Modal training setup and checkpoint flow: https://qdrant.tech/articles/sparse-embeddings-ecommerce-part-2/
- **Article Part 3:** evaluation and hard negatives: https://qdrant.tech/articles/sparse-embeddings-ecommerce-part-3/
- **Main lab:** `notebooks/lab.ipynb` shows Qdrant indexing, querying, hybrid search, and full 2K eval.
- **Released model:** https://huggingface.co/thierrydamiba/splade-ecommerce-esci
- **Sentence Transformers sparse encoder docs:** https://sbert.net/docs/sparse_encoder/training_overview.html

The production move is not just "train a model." It is: measure a baseline, train on domain labels, inspect failures, evaluate on held-out queries, then decide how to serve the signal.